# o1简化实现版本
本教程主要实现了一个简化版本的o1模型，便于学习，能够进行简单的算术推理。

本教程包括以下内容：  
1. 详细的架构描述，重点介绍了实现高级推理能力的新颖组件。  
2. 训练方法，结合了监督学习、强化学习以及推理标记的创新使用。  
3. 高级推理策略，包括自适应推理和扩展上下文窗口的高效管理。  
4. 实验结果，展示了模型在各种推理基准测试中的性能表现。  
5. 对模型能力、当前局限性以及未来潜在发展方向的讨论。

## 背景介绍
### o1之前模型的局限
尽管像`GPT-3`和`GPT-4`这样的大语言模型（LLMs）展示了令人印象深刻的能力，但它们在需要多步推理、逻辑一致性和错误纠正的任务上往往表现不佳。这些局限性源于以下几个方面：

1. **浅层推理深度**：传统模型在单次推理中生成响应，缺乏迭代优化。  
2. **缺乏自我纠正能力**：模型无法自主检测并纠正其错误。  
3. **静态推理过程**：推理机制不会根据输入的复杂性或生成输出的正确性进行调整。  
4. **有限的上下文理解能力**：尽管具有较大的上下文窗口，模型在处理长输入时往往难以保持连贯性和相关性。

### o1系列模型的改进
o1 模型系列基于该领域的几项关键进展：  
1. **Chain-of-Thought (CoT) Reasoning**：使模型能够生成并利用中间推理步骤。  
2. **Reinforcement Learning from Human Feedback (RLHF)**：使用强化学习根据人类偏好和反馈对模型进行微调。  
3. **Process Supervision**：对中间推理步骤提供反馈，而不仅仅是最终输出。  
4. **Tree of Thoughts**：同时探索多条推理路径。  
5. **Large Context Windows**：扩展上下文窗口以支持更复杂的推理。

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import math
import random

# Constants
CONTEXT_WINDOW_SIZE = 128000
MAX_OUTPUT_TOKENS_PREVIEW = 32768
MAX_OUTPUT_TOKENS_MINI = 65536

# Set random seeds for reproducibility
torch.manual_seed(0)
random.seed(0)


## 词表和Tokenize函数

In [13]:
# Enhanced vocabulary
vocab = {
    '<pad>': 0, '<sos>': 1, '<eos>': 2, 'Step:': 3, '+': 4, '-': 5, '*': 6, '/': 7, '=': 8,
    '0': 9, '1': 10, '2': 11, '3': 12, '4': 13, '5': 14, '6': 15, '7': 16, '8': 17, '9': 18,
    'if': 19, 'then': 20, 'else': 21, 'greater': 22, 'less': 23, 'equal': 24,
    'Calculate': 25, 'the': 26, 'sum': 27, 'of': 28, 'and': 29,
    'difference': 30, 'between': 31, 'product': 32, 'quotient': 33,
    'First,': 34, 'Next,': 35, 'Finally,': 36, 'result': 37, 'is': 38,
    '<subtask>': 39  # New token for subtask generation
}
vocab_size = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}


def tokenize(text):
    return [vocab.get(token, vocab['<pad>']) for token in text.strip().split()]

def detokenize(indices):
    return ' '.join([inv_vocab.get(idx, ' ') for idx in indices])

# O1模型架构


## 核心组成部分
o1 模型架构由以下关键组件组成：  
1. **Embedding Layer**：将输入的 token 转换为连续的向量表示。  
2. **Transformer Encoder-Decoder**：处理输入序列并生成输出，支持长上下文和复杂结构的处理。  
3. **Chain-of-Thought Module**：生成中间推理步骤，保持连贯性和逻辑性。  
4. **Reasoning Token Generator**：生成内部的“推理 token”，这些 token 表示模型的思考过程，但不会出现在最终输出中。  
5. **Policy and Value Networks**：在强化学习框架中使用，用于估计动作值并指导决策。


![alt text](./_img/o1_frame.png)

### 位置编码

In [14]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


### Transformer Block
o1 模型的核心基于 Transformer 架构，类似于 GPT-3 和 GPT-4 中使用的架构。其关键组件包括：  
1. **Multi-head Self-Attention（多头自注意力机制）**：使模型能够同时关注输入序列的不同部分，从而捕捉复杂的依赖关系。  
2. **Feed-forward Neural Networks（前馈神经网络）**：处理注意力机制的输出并引入非线性。  
3. **Layer Normalization and Residual Connections（层归一化和残差连接）**：稳定训练过程并支持更深层次的网络结构。

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super(TransformerBlock, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Ensure x has the correct shape (batch_size, seq_len, d_model)
        if x.dim() == 2:
            x = x.unsqueeze(0)  # Add batch dimension if missing
        elif x.dim() == 4:
            x = x.squeeze(2)  # Remove extra dimension if present
        
        attn_output, _ = self.self_attn(x, x, x)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)
        ff_output = self.feed_forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm2(x)
        return x


### O1模型结构

#### 1. Embedding
```python
self.embed = nn.Embedding(vocab_size, d_model)
```
- **作用**：  
  嵌入层将每个词汇映射为一个固定维度的稠密向量。这是深度学习模型中常用的技巧，目的是将离散的词汇索引转换成具有上下文关系的连续向量表示。
- **输入**：  
  - `vocab_size`：词汇表的大小，表示模型能够识别的不同单词的数量。
  - `d_model`：嵌入的维度，表示每个词汇映射到的向量空间的维度。
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, d_model)` 的张量，每个输入词汇被转换为一个 `d_model` 维度的向量。

#### 2. Positional Encoding
```python
self.pos_encoder = PositionalEncoding(d_model)
```
- **作用**：  
  由于Transformer架构本身没有任何序列顺序信息，位置编码被用来向模型输入中添加关于单词在序列中位置的信息。通过在词嵌入中加上位置编码，模型能够理解单词顺序和位置关系。
  
  `PositionalEncoding` 类负责根据给定的维度 `d_model` 生成位置编码，通常通过正弦和余弦函数来实现，使得位置编码能够在不同长度的序列中有不同的表示。
  
- **输入**：  
  - `d_model`：嵌入向量的维度。
  
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, d_model)` 的张量，包含了每个位置的编码信息。

#### 3. Transformer Layers
```python
self.transformer_layers = nn.ModuleList([TransformerBlock(d_model, nhead) for _ in range(num_layers)])
```
- **作用**：  
  这一部分是模型的核心部分，由多个Transformer编码层（`TransformerBlock`）构成，每个编码层都包含自注意力机制（self-attention）和前馈神经网络。每一层都能从输入中学习到更加复杂的特征表示。
  
  - `TransformerBlock` 是一个模块化的变换器层，包含自注意力层（Multi-Head Attention）和前馈网络（Feed-Forward Network）。
  - `num_layers`：指模型的变换器层数，通常更多的层数意味着模型可以学习到更复杂的特征。
  - `nhead`：每层自注意力机制的头数，影响每个变换器层中自注意力的计算方式。更多的头数有助于模型从不同的角度理解输入。

- **输入**：  
  - `d_model`：每个词的嵌入维度。
  - `nhead`：自注意力机制的头数。
  
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, d_model)` 的张量，表示每一层的输出特征。

#### 4. Completion Decoder
```python
self.completion_decoder = nn.Linear(d_model, vocab_size)
```
- **作用**：  
  该解码器用于生成任务的最终输出，也就是完成推理步骤之后的输出。通过对`Transformer`的输出应用一个全连接层，将其映射到词汇表大小（`vocab_size`）上，得到每个词汇的概率分布。最终，模型根据这个分布生成任务的完成令牌。
  
- **输入**：  
  - `d_model`：变换器层输出的特征维度。
  
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, vocab_size)` 的张量，表示每个位置生成词汇的概率分布。

#### 5. Reasoning Decoder
```python
self.reasoning_decoder = nn.Linear(d_model, vocab_size)
```
- **作用**：  
    o1系列的一个关键创新是引入了推理标记（reasoning tokens）。这些标记使模型能够在内部进行“思考”，分解其对提示的理解，并考虑生成响应的多种方法。推理标记生成器生成这些内部标记，这些标记由模型处理，但不包含在最终可见的输出中。

    推理token的关键特征：  
    1. **内部表示**：推理token代表了模型的内部思考过程。  
    2. **可丢弃性**：在生成最终输出后，推理token会被丢弃，不会保留在上下文中以供后续交互使用。  
    3. **资源消耗**：尽管在输出中不可见，推理token会占用模型上下文窗口的空间，并作为输出token计费。
    
    推理解码器的作用与完成解码器类似，不过它是专门用于生成推理过程中的推理步骤。通过将变换器的输出映射到词汇表大小上，模型能够生成推理过程中的相关令牌。
  
- **输入**：  
  - `d_model`：变换器输出的特征维度。
  
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, vocab_size)` 的张量，表示每个位置生成推理步骤的概率分布。

#### 6. 强化学习机制（Reinforcement Learning Mechanisms）

将RL集成到o1模型架构中可以实现以下功能：  
1. **动态决策**：模型可以根据环境反馈或内部评估调整其推理过程。  
2. **策略优化**：调整模型参数以最大化预期奖励，从而随着时间的推移提升性能（Schulman et al., 2017）。  
3. **价值估计**：评估不同推理路径的潜在未来奖励，以指导行动选择。

RL组件通过额外的策略和价值头实现：

In [16]:
class PolicyValueHeads(nn.Module):
    def __init__(self, d_model, vocab_size):
        super(PolicyValueHeads, self).__init__()
        self.policy_head = nn.Linear(d_model, vocab_size)
        self.value_head = nn.Linear(d_model, 1)

    def forward(self, x):
        policy_logits = self.policy_head(x)
        values = self.value_head(x).squeeze(-1)
        return policy_logits, values

#### 7. Subtask Head
```python
self.subtask_head = nn.Linear(d_model, 1)
```
- **作用**：  
  子任务头用于预测当前生成步骤是否需要创建一个新的子任务。该模块输出一个值，用于判断是否需要继续生成任务或开始一个子任务。
  
- **输入**：  
  - `d_model`：变换器输出的特征维度。
  
- **输出**：  
  - 一个形状为 `(batch_size, seq_len, 1)` 的张量，表示每个生成步骤是否需要一个子任务。

In [17]:
class O1Model(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, is_mini=False):
        super(O1Model, self).__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer_layers = nn.ModuleList([TransformerBlock(d_model, nhead) for _ in range(num_layers)])
        self.reasoning_decoder = nn.Linear(d_model, vocab_size)
        # self.completion_decoder = nn.Linear(d_model, vocab_size)
        # self.value_head = nn.Linear(d_model, 1)
        self.policy_value_heads = PolicyValueHeads(d_model, vocab_size)
        self.subtask_head = nn.Linear(d_model, 1)
        self.is_mini = is_mini
        self.max_reasoning_tokens = 1000

    def forward(self, src, reasoning_tokens=None, generate_reasoning=True):
        if src.dim() == 1:
            src = src.unsqueeze(0)
        elif src.dim() == 3:
            src = src.squeeze(1)
        
        if src.size(1) == 0:
            print(f"Warning: Empty input tensor in forward pass. Shape: {src.shape}")
            batch_size = src.size(0)
            return torch.zeros(batch_size, 1, self.vocab_size), torch.zeros(batch_size, 1, self.vocab_size), torch.zeros(batch_size, 1)
        
        src = self.embed(src)
        if reasoning_tokens is not None:
            reasoning_embeddings = self.embed(reasoning_tokens)
            src = torch.cat([src, reasoning_embeddings], dim=1)
        
        src = self.pos_encoder(src)
        
        for layer in self.transformer_layers:
            src = layer(src)
        
        # completion_logits = self.completion_decoder(src)
        # values = self.value_head(src).squeeze(-1)
        completion_logits, values = self.policy_value_heads(src)
        
        if generate_reasoning:
            reasoning_logits = self.reasoning_decoder(src)
            return completion_logits, reasoning_logits, values
        else:
            return completion_logits, values

    def generate_completion(self, input_ids, max_new_tokens, num_paths=3):
        max_tokens = MAX_OUTPUT_TOKENS_MINI if self.is_mini else MAX_OUTPUT_TOKENS_PREVIEW
        max_new_tokens = min(max_new_tokens, max_tokens)
        
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        elif input_ids.dim() == 3:
            input_ids = input_ids.squeeze(1)
        
        paths = []
        for _ in range(num_paths):
            generated = input_ids.clone()
            reasoning_tokens = torch.tensor([], dtype=torch.long, device=input_ids.device)
            completion_tokens = []
            subtasks = []
            
            for _ in range(max_new_tokens):
                if generated.size(1) + reasoning_tokens.size(0) >= CONTEXT_WINDOW_SIZE:
                    break
                
                completion_logits, reasoning_logits, values = self(generated, reasoning_tokens)
                
                if completion_logits.numel() == 0:
                    print(f"Warning: completion_logits is empty. Input shape: {generated.shape}")
                    break
                
                next_token_logits = completion_logits[:, -1, :]
                next_token = self.sample_token(next_token_logits)
                
                reasoning_token = self.sample_token(reasoning_logits[:, -1, :])
                reasoning_tokens = torch.cat([reasoning_tokens, reasoning_token.unsqueeze(0)])
                
                if reasoning_tokens.size(0) > self.max_reasoning_tokens:
                    reasoning_tokens = reasoning_tokens[-self.max_reasoning_tokens:]
                
                last_hidden = self.embed(generated[:, -1])
                subtask_prob = torch.sigmoid(self.subtask_head(last_hidden))
                if subtask_prob > 0.5:
                    subtask = self.generate_subtask(generated, reasoning_tokens)
                    subtasks.append(subtask)
                    generated = torch.cat([generated, torch.tensor([[vocab['<subtask>']]]).to(generated.device)], dim=1)
                else:
                    generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
                    completion_tokens.append(next_token.item())
                
                if self.should_revise_reasoning():
                    generated, reasoning_tokens = self.revise_reasoning(generated, reasoning_tokens)
                
                if next_token.item() == vocab['<eos>']:
                    break
            
            paths.append((completion_tokens, reasoning_tokens.tolist(), subtasks))
        
        if not paths:
            print("Warning: No valid paths generated")
            return [], [], []
        
        rewards = [self.compute_reward(p[0], p[1], p[2]) for p in paths]
        best_path = paths[rewards.index(max(rewards))]
        
        return best_path[0], best_path[1], best_path[2]

    def sample_token(self, logits, temperature=0.7):
        probs = F.softmax(logits / temperature, dim=-1)
        return torch.multinomial(probs, 1).squeeze(-1)

    def add_reasoning_token(self, token):
        self.reasoning_buffer.append(token)
        if len(self.reasoning_buffer) > self.max_reasoning_tokens:
            self.reasoning_buffer.pop(0)

    def should_revise_reasoning(self):
        # Implement logic to decide if reasoning should be revised
        return random.random() < 0.1  # 10% chance of revision for demonstration

    def revise_reasoning(self, generated, reasoning_tokens):
        # Implement logic to revise reasoning
        # For demonstration, we'll just remove the last few tokens from both
        return generated[:, :-5], reasoning_tokens[:-5]

    def generate_subtask(self, context, reasoning_tokens):
        subtask_tokens = []
        for _ in range(20):  # Max subtask length
            logits, _, _ = self(context, reasoning_tokens)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)
            subtask_tokens.append(next_token.item())
            context = torch.cat([context, next_token.unsqueeze(1)], dim=1)
            if next_token.item() == vocab['<eos>']:
                break
        return subtask_tokens

    def compute_reward(self, completion_tokens, reasoning_tokens, subtasks):
        completion_reward = len(completion_tokens) * 0.1
        reasoning_reward = len(set(reasoning_tokens)) * 0.2
        subtask_reward = len(subtasks) * 0.5
        coherence_reward = self.compute_coherence(completion_tokens)
        process_reward = self.compute_process_reward(reasoning_tokens)
        return completion_reward + reasoning_reward + subtask_reward + coherence_reward + process_reward

    def compute_coherence(self, tokens):
        # Simple coherence check (can be made more sophisticated)
        return sum(1 for i in range(len(tokens)-1) if tokens[i] + 1 == tokens[i+1]) * 0.1

    def compute_process_reward(self, reasoning_tokens):
        # Implement a more sophisticated process reward
        unique_tokens = len(set(reasoning_tokens))
        return unique_tokens * 0.1  # Reward diverse reasoning


### `forward`函数执行过程

`forward` 函数是 `O1Model` 类中的核心部分，它定义了模型在输入数据时如何进行前向传播，生成输出。在深度学习模型中，`forward` 函数将输入的特征传递给模型的各个层，并最终返回结果。

#### 1. 生成模拟输入

In [ ]:
import torch

# 假设参数
batch_size = 2
seq_len = 5
reasoning_seq_len = 3
vocab_size = 100
d_model = 128
num_layers = 4
nhead = 8
dropout = 0.1

embed = nn.Embedding(vocab_size, d_model)
pos_encoder = PositionalEncoding(d_model)
transformer_layers = nn.ModuleList([TransformerBlock(d_model, nhead) for _ in range(num_layers)])
completion_decoder = nn.Linear(d_model, vocab_size)
reasoning_decoder = nn.Linear(d_model, vocab_size)
policy_value_heads = PolicyValueHeads(d_model, vocab_size)
subtask_head = nn.Linear(d_model, 1)
is_mini = False
max_reasoning_tokens = 1000
max_new_tokens = 20  # 最大生成的token数量
num_paths = 3  # 生成3条路径

# 模拟的输入数据
# 1. 主任务输入（src），形状为 (batch_size, seq_len)
src = torch.randint(0, vocab_size, (batch_size, seq_len))

# 2. 推理令牌（reasoning_tokens），形状为 (batch_size, reasoning_seq_len)
reasoning_tokens = torch.randint(0, vocab_size, (batch_size, reasoning_seq_len))

# 3. generate_reasoning 控制是否生成推理
generate_reasoning = True

# 输出模拟的输入
print("src (main task input):", src)
print("reasoning_tokens (reasoning task input):", reasoning_tokens)
print("generate_reasoning:", generate_reasoning)


#### 1. 输入维度检查及空值处理
检查输入 `src` 的维度。如果 `src` 的维度是 1，表示只有一个句子或序列，因此需要增加一个 batch 维度（通过 `unsqueeze(0)`）。如果 `src` 是一个形状为 `(batch_size, 1, seq_len)` 的三维张量，则需要去掉第二维（通过 `squeeze(1)`）。这确保了输入的维度一致性。

检查输入序列 `src` 是否为空。如果输入为空（即 `src.size(1) == 0`），模型返回形状为 `(batch_size, 1, vocab_size)` 的零张量，并发出警告。这是为了防止空输入导致后续操作出错。

In [ ]:
print(src.dim())
if src.dim() == 1:
    src = src.unsqueeze(0)
elif src.dim() == 3:
    src = src.squeeze(1)
if src.size(1) == 0:
    print(f"Warning: Empty input tensor in forward pass. Shape: {src.shape}")
    batch_size = src.size(0)
    print(torch.zeros(batch_size, 1, vocab_size), torch.zeros(batch_size, 1, vocab_size), torch.zeros(batch_size, 1))

#### 3. 词嵌入（Embedding）
- **作用**：将输入的词汇索引 `src` 映射到词嵌入空间，得到形状为 `(batch_size, seq_len, d_model)` 的张量，其中 `d_model` 是嵌入的维度。通过嵌入层，模型将每个单词映射为一个向量，这样可以处理文本数据中的语义信息。


In [ ]:
print(f"Before Embedding: {src.shape}")
src = embed(src)
print(f"After Embedding: {src.shape}")

#### 4. 添加推理标记Reasoning Tokens
- **作用**：如果提供了 `reasoning_tokens`，则将其嵌入到 `d_model` 维度的空间中。然后将推理令牌的嵌入与原始输入 `src` 在序列维度（`dim=1`）上拼接。这允许模型同时处理任务的输入和推理过程中的令牌。

In [ ]:
if reasoning_tokens is not None:
    print(f"Before Adding (Reasoning Tokens): {src.shape}")
    print(f"Reasoning Tokens Shape: {reasoning_tokens.shape}")
    reasoning_embeddings = embed(reasoning_tokens)
    src = torch.cat([src, reasoning_embeddings], dim=1)
    print(f"After Adding (Reasoning Tokens): {src.shape}")

#### 5. 位置编码
将位置编码添加到 `src` 中。位置编码用于表示每个词汇在输入序列中的相对位置，帮助模型理解序列中的顺序关系。输出的 `src` 仍然保持形状 `(batch_size, seq_len, d_model)`。

In [ ]:
src = pos_encoder(src)

#### 6. Transformer Layers
对嵌入后的输入进行一系列变换器层的处理。每个变换器层包括自注意力机制和前馈神经网络。`src` 逐层传递并得到更加抽象的特征表示。每一层的输出都会作为下一层的输入。

> **注意**：`self.transformer_layers` 是一个 `ModuleList`，包含了 `num_layers` 个 `TransformerBlock`。每个 `TransformerBlock` 负责对输入进行变换，捕捉更深层次的模式和上下文信息。

In [ ]:
for layer in transformer_layers:
    src = layer(src)

#### 7. 添加强化学习

In [ ]:
completion_logits, values = policy_value_heads(src)
print(f"Completion Logits Shape: {completion_logits.shape}")
print(completion_logits)
print(f"Values Shape: {values.shape}")
print(values)

#### 9. 生成推理
根据 `generate_reasoning` 参数决定是否生成推理相关的输出。如果为 `True`，则通过 `reasoning_decoder` 生成推理步骤的 logits，并与任务完成的 logits 以及值一起返回。如果为 `False`，则只返回任务完成的 logits 和对应的值。

- **输出**：
  - 如果 `generate_reasoning` 为 `True`，返回三个张量：
    - `completion_logits`：生成任务完成的概率分布，形状为 `(batch_size, seq_len, vocab_size)`。
    - `reasoning_logits`：生成推理步骤的概率分布，形状为 `(batch_size, seq_len, vocab_size)`。
    - `values`：每个生成位置的值，形状为 `(batch_size, seq_len)`。
  - 如果 `generate_reasoning` 为 `False`，只返回：
    - `completion_logits`：生成任务完成的概率分布。
    - `values`：每个生成位置的值。

In [ ]:

if generate_reasoning:
    reasoning_logits = reasoning_decoder(src)
    print(completion_logits, reasoning_logits, values)
    # return completion_logits, reasoning_logits, values

else:
    print(completion_logits, values)
    # return completion_logits, values

### `generate_completion`过程详细解读

`generate_completion` 函数的主要作用是基于初始输入 `input_ids` 生成一系列后续的文本（任务完成的内容），并根据生成的多个“路径”进行选择，最后返回最优的完成结果。


### 输入参数

1. **`input_ids`**:
   - 形状：`(batch_size, seq_len)`，表示模型的初始输入序列（通常是任务的开始部分）。
   - 作用：这是模型生成任务的起始点，通常是一个词汇索引的张量。

2. **`max_new_tokens`**:
   - 类型：`int`，表示模型最多生成的词汇数量。
   - 作用：控制生成的序列的最大长度。它会与 `max_tokens` 限制相比较，确保不超过生成的最大 token 数量。

3. **`num_paths`**:
   - 类型：`int`，默认为 `3`，表示生成的路径数量。
   - 作用：生成多个不同的“路径”，每个路径代表生成序列的一个可能的输出。最终会选择最优的路径返回。

#### 1. 输入控制
- 根据模型是否为“迷你版” (`self.is_mini`)，决定生成文本的最大 token 数量。`MAX_OUTPUT_TOKENS_MINI` 和 `MAX_OUTPUT_TOKENS_PREVIEW` 是预先定义的常量，代表最大生成 token 数量的上限。

- 检查 `input_ids` 的维度，并根据需要调整。若输入是 1 维的，增加一个 batch 维度；若输入是 3 维的，去除多余的维度。确保输入的维度适合模型的要求。

In [ ]:
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
print(input_ids)

In [ ]:
max_tokens = MAX_OUTPUT_TOKENS_MINI if is_mini else MAX_OUTPUT_TOKENS_PREVIEW
max_new_tokens = min(max_new_tokens, max_tokens)

if input_ids.dim() == 1:
    input_ids = input_ids.unsqueeze(0)
elif input_ids.dim() == 3:
    input_ids = input_ids.squeeze(1)

#### 2. 路径生成

```python
paths = []
for _ in range(num_paths):
    generated = input_ids.clone()
    reasoning_tokens = torch.tensor([], dtype=torch.long, device=input_ids.device)
    completion_tokens = []
    subtasks = []
```
从这段代码开始，主要是生成`num_paths`条推理路径，每个路径表示一个可能的生成序列。初始化每个路径时，`generated` 是一个克隆的 `input_ids`，并为每个路径初始化 `reasoning_tokens`、`completion_tokens` 和 `subtasks`。

下面只模拟一条路径的生成过程。

In [ ]:
paths = []

In [ ]:
generated = input_ids.clone()
reasoning_tokens = torch.tensor([], dtype=torch.long, device=input_ids.device)
completion_tokens = []
subtasks = []

#### 3. 开始生成路径


In [ ]:
model = O1Model(vocab_size, d_model, nhead, num_layers, is_mini)

In [ ]:
for _ in range(max_new_tokens):
    if generated.size(1) + reasoning_tokens.size(0) >= CONTEXT_WINDOW_SIZE:
        break
    
    # 每次生成时，调用 `self(generated, reasoning_tokens)`，返回 `completion_logits`、`reasoning_logits` 和 `values`。
    completion_logits, reasoning_logits, values = model(generated, reasoning_tokens)

    if completion_logits.numel() == 0:
        print(f"Warning: completion_logits is empty. Input shape: {generated.shape}")
        break
    
    # 从 `completion_logits` 和 `reasoning_logits` 中选择下一个 token。`sample_token` 使用 softmax 输出概率分布，随机选择下一个词汇。这是为了生成多样的结果。
    next_token_logits = completion_logits[:, -1, :]
    next_token = model.sample_token(next_token_logits)
    
    reasoning_token = model.sample_token(reasoning_logits[:, -1, :])
    reasoning_tokens = torch.cat([reasoning_tokens, reasoning_token.unsqueeze(0)])
    
    # 确保推理令牌 `reasoning_tokens` 不超过最大长度 `max_reasoning_tokens`，如果超出则截取最后的令牌。
    if reasoning_tokens.size(0) > model.max_reasoning_tokens:
        reasoning_tokens = reasoning_tokens[-model.max_reasoning_tokens:]
    
    # 模型根据 `subtask_prob` 判断是否生成子任务。如果 `subtask_prob > 0.5`，则生成一个子任务。生成子任务时，通过 `self.generate_subtask(generated, reasoning_tokens)` 生成与当前序列相关的子任务。子任务被附加到 `subtasks` 列表中，并将子任务标记 `<subtask>` 添加到 `generated` 中。
    last_hidden = model.embed(generated[:, -1])
    subtask_prob = torch.sigmoid(model.subtask_head(last_hidden))
    if subtask_prob > 0.5:
        subtask = model.generate_subtask(generated, reasoning_tokens)
        subtasks.append(subtask)
        generated = torch.cat([generated, torch.tensor([[vocab['<subtask>']]]).to(generated.device)], dim=1)
    else:
        generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
        completion_tokens.append(next_token.item())
    
    # 在每次生成后，模型有一定的概率（`should_revise_reasoning()`）决定是否修正推理过程。修正方式可以是删除或修改推理令牌。
    if model.should_revise_reasoning():
        generated, reasoning_tokens = model.revise_reasoning(generated, reasoning_tokens)
    
    # 如果生成的下一个 token 是 `<eos>`（即序列结束符），则停止生成过程。
    if next_token.item() == vocab['<eos>']:
        break

In [ ]:
paths.append((completion_tokens, reasoning_tokens.tolist(), subtasks))

#### 计算奖励并选择最优路径

In [ ]:
rewards = [model.compute_reward(p[0], p[1], p[2]) for p in paths]
best_path = paths[rewards.index(max(rewards))]

## PPO算法

### PPO 算法的原理及公式

Proximal Policy Optimization (PPO) 是一种策略优化方法，它基于**策略梯度**（Policy Gradient）方法，并通过引入**剪切**（clipping）技巧来增强训练过程的稳定性。PPO 是一种增强版的 **TRPO**（Trust Region Policy Optimization），其核心思想是限制每次策略更新的幅度，以确保训练过程的稳定性。

#### 1. **策略梯度方法（Policy Gradient）**

在强化学习中，策略梯度方法的目标是通过最大化一个特定的目标函数来优化策略。假设当前的策略是 $\pi_{\theta}(a|s)$，即给定状态 $s$，策略 $\pi_{\theta}$ 会输出采取动作 $a$ 的概率。

策略梯度的目标是最大化 **期望回报**（Expected Return）：

$$
J(\theta) = \mathbb{E}_{\pi_{\theta}}[R] = \mathbb{E}_{\pi_{\theta}} \left[ \sum_{t=0}^{T} \gamma^t r_t \right]
$$

其中，$r_t$ 是在时间步 $t$ 的奖励，$\gamma$ 是折扣因子，$T$ 是终止时间步。要最大化期望回报，我们需要计算该期望的梯度：

$$
\nabla_{\theta} J(\theta) = \mathbb{E}_{\pi_{\theta}} \left[ \nabla_{\theta} \log \pi_{\theta}(a_t | s_t) R_t \right]
$$

其中，$\log \pi_{\theta}(a_t | s_t)$ 是策略对数概率，$R_t$ 是从时间步 $t$ 开始的累计回报。

#### 2. **PPO算法的核心：**

PPO 通过引入 **剪切（Clipping）** 技术，限制了每次更新策略的幅度，避免了策略的过度更新，从而提高了训练的稳定性。

##### 2.1 **代理目标函数（Surrogate Objective Function）**

PPO 提出了一个“代理目标函数”来优化策略，这个目标函数包括一个比率项，表示当前策略与旧策略之间的比率。PPO 的代理目标函数通常写作：

$$
L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t \right) \right]
$$

其中：
- $r_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ 是新旧策略的概率比率。
- $A_t$ 是优势函数（Advantage Function），表示某一动作的相对好坏。
- $\epsilon$ 是剪切参数，通常设为 0.1 或 0.2。

这个目标函数的核心思想是：在策略更新时，如果策略变化太大，计算出的目标值将会被剪切，从而避免了过度更新导致的不稳定性。

##### 2.2 **优势函数 $A_t$**

优势函数 $A_t$ 用来衡量某一动作的质量相对于基准动作的优势。在 PPO 中，优势函数通常通过 **广义优势估计（GAE，Generalized Advantage Estimation）** 进行计算，GAE 结合了 **时间差分误差**（TD-error）和 **回报估计** 来进行平滑：

$$
A_t = \delta_t + (\gamma \lambda) \delta_{t+1} + (\gamma \lambda)^2 \delta_{t+2} + \cdots
$$

其中，$\delta_t$ 是时间差分误差，计算公式为：

$$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

这里：
- $r_t$ 是奖励，
- $V(s_t)$ 是状态值函数的预测。

$\gamma$ 是折扣因子，$\lambda$ 是用于平滑的超参数。

##### 2.3 **剪切（Clipping）机制**

PPO 引入的剪切机制是为了防止新旧策略的比率发生过大变化。通过引入一个 $\text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon)$ 操作，如果比率 $r_t(\theta)$ 超过了 $1 \pm \epsilon$ 范围，目标函数就会被限制，防止策略更新过大。

这使得 PPO 可以在没有复杂的二阶优化信息的情况下进行稳定的策略优化，相比于 TRPO，PPO 更加高效且易于实现。

##### 2.4 **最终的目标函数**

PPO 最终的目标函数就是我们上面提到的代理目标函数，再加上一个价值函数损失项（critic loss）和熵损失项（entropy loss），从而构建最终的优化目标：

$$
L^{PPO}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t \right) \right] - c_1 \mathbb{E}_t \left[ (V_{\theta}(s_t) - R_t)^2 \right] + c_2 \mathbb{E}_t \left[ H(\pi_{\theta}(\cdot|s_t)) \right]
$$

其中：
- $c_1$ 是价值损失项的系数，
- $c_2$ 是熵损失项的系数，
- $H(\pi_{\theta}(\cdot|s_t))$ 是策略的熵，鼓励策略保持足够的随机性，从而避免过早收敛。

### 3. **总结公式**

PPO 算法的核心公式包括了一个代理目标函数：

$$
L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t \right) \right]
$$

优势函数的计算通过 GAE 方法：

$$
A_t = \delta_t + (\gamma \lambda) \delta_{t+1} + (\gamma \lambda)^2 \delta_{t+2} + \cdots
$$

剪切机制限制了策略更新幅度，最终目标函数加上了价值函数和熵的正则化项，构成了最终的优化目标：

$$
L^{PPO}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t \right) \right] - c_1 \mathbb{E}_t \left[ (V_{\theta}(s_t) - R_t)^2 \right] + c_2 \mathbb{E}_t \left[ H(\pi_{\theta}(\cdot|s_t)) \right]
$$

这个目标函数通过保证策略更新的平稳性、提高探索性和准确性，从而使 PPO 成为一个强大且稳定的强化学习算法。

### PPO 算法实现类 `PPO`

下面这个代码实现了 `PPO` 算法的训练过程，提供了计算优势函数、更新模型参数以及优化策略等功能。

#### 类的成员变量
- `model`：策略网络模型，通常是一个深度神经网络，用于根据状态输出动作分布。
- `optimizer`：优化器，用于更新模型参数，通常是 Adam 或其他优化算法。
- `clip_epsilon`：剪切的阈值，控制新旧策略之间允许的差异范围。通常设为 0.2，这意味着策略更新的比率超过 1 ± 0.2 时会被裁剪。
- `value_coef`：价值函数损失的系数，用于平衡策略优化和价值函数优化之间的权重。
- `entropy_coef`：熵损失的系数，用于引导策略更加随机（增加探索），防止策略收敛过早。

### 每个模块的作用

1. **`compute_advantages` 函数：**
   该函数计算优势函数（Advantage Function），用于评估当前状态下的动作相对于基准动作的优势。它基于奖励信号和价值函数来估计每个时刻的优势，并通过 GAE（Generalized Advantage Estimation）方法来进行平滑和加权。

2. **`update` 函数：**
   该函数是 PPO 算法的核心，负责执行策略优化过程。它利用策略网络和价值网络来计算当前策略的优势，计算损失函数（包括演员损失、评论员损失和熵损失），并通过反向传播更新模型的参数。

### 函数解读

#### `compute_advantages` 函数

- **输入：**
  - `rewards`：当前环境中执行动作后的奖励序列。
  - `values`：当前模型预测的状态值函数。
  - `gamma`：折扣因子，决定未来奖励对当前决策的影响。
  - `lambda_`：用于 GAE 的参数，控制优势的平滑程度。
  
- **输出：**
  - `advantages`：每个时间步的优势值，表示某个动作相对于基准动作的优劣。
  - `returns`：累积奖励（回报），即未来奖励的折扣加权和。

#### `update` 函数

- **输入：**
  - `states`：当前的状态序列。
  - `actions`：在这些状态下采取的动作。
  - `old_log_probs`：旧策略下的动作的对数概率。
  - `rewards`：当前环境中的奖励信号。
  - `old_values`：旧策略下的状态值估计。

- **输出：**
  - 无输出，该函数直接更新模型参数。

##### 函数内部步骤

1. **状态的维度调整：**
   如果状态的维度是 2D（即只有批量大小和序列长度），将状态扩展成 3D 以适应模型输入。
   
2. **计算优势和回报：**
   通过调用 `compute_advantages` 函数来计算优势函数和回报。这里的优势函数是通过奖励信号和状态值函数来估计的，采用 GAE 方法。

3. **PPO 训练过程：**
   在训练过程中，使用 PPO 中的一个常见技巧，即多次迭代优化（5次 PPO 迭代）。每次迭代会根据当前策略计算新的动作概率，并计算损失。

4. **损失函数的计算：**
   - **Actor Loss：** 通过计算新旧策略比率来衡量策略更新的效果，使用 `torch.min` 保证更新不偏离原有策略太远。
   - **Critic Loss：** 使用均方误差损失函数（`MSELoss`）来计算当前策略的价值估计与真实回报之间的误差。
   - **Entropy Loss：** 计算策略的熵（即不确定度），并希望策略保持足够的随机性，以便探索更多的可能性。

5. **更新优化器：**
   计算总损失并使用反向传播（`backward`）来更新模型参数。使用优化器（`optimizer.step()`）来执行梯度下降。

### 总结

这段代码实现了 PPO 算法的核心思想，并通过定义一个类 `PPO` 来组织代码。它利用模型的策略网络和价值网络，通过计算优势函数、回报、演员损失和评论员损失，进行策略优化。每一次训练迭代都通过反向传播来更新模型的参数，并最终通过优化器来调整策略和价值函数。

In [18]:
class PPO:
    def __init__(self, model, optimizer, clip_epsilon=0.2, value_coef=0.5, entropy_coef=0.01):
        self.model = model
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

    def compute_advantages(self, rewards, values, gamma=0.99, lambda_=0.95):
        advantages = torch.zeros_like(rewards)
        last_advantage = 0
        
        # Make sure to only iterate through the valid range
        for t in reversed(range(len(rewards))):
            if t + 1 < len(values):
                delta = rewards[t] + gamma * values[t + 1] - values[t]
            else:
                delta = rewards[t] - values[t]
                
            advantages[t] = delta + gamma * lambda_ * last_advantage
            last_advantage = advantages[t]
        
        returns = advantages + values[:len(advantages)]
        return advantages, returns

    def update(self, states, actions, old_log_probs, rewards, old_values):
        # Reshape states if necessary
        if states.dim() == 2:
            batch_size, seq_len = states.shape
            states = states.unsqueeze(0)  # Add a dimension to make it [1, batch_size, seq_len]
        else:
            num_steps, batch_size, seq_len = states.shape
        
        # Flatten other tensors
        actions_flat = actions.view(-1)
        old_log_probs_flat = old_log_probs.view(-1)
        advantages, returns = self.compute_advantages(rewards, old_values)
        advantages_flat = advantages.view(-1)
        returns_flat = returns.view(-1)
        
        for _ in range(5):  # PPO epochs
            logits, _, values = self.model(states.view(-1, seq_len))
            
            # Focus on the logits of the last token in the sequence
            next_token_logits = logits[:, -1, :]
            new_probs = F.softmax(next_token_logits, dim=-1)
            dist = Categorical(new_probs)
            
            # Ensure actions_flat matches the shape of new_probs
            actions_flat_truncated = actions_flat[:new_probs.size(0)]
            old_log_probs_flat_truncated = old_log_probs_flat[:new_probs.size(0)]
            advantages_flat_truncated = advantages_flat[:new_probs.size(0)]
            returns_flat_truncated = returns_flat[:new_probs.size(0)]
            
            # Calculate new log probabilities
            new_log_probs = dist.log_prob(actions_flat_truncated)
            
            # Calculate probability ratio
            ratio = torch.exp(new_log_probs - old_log_probs_flat_truncated)
            surr1 = ratio * advantages_flat_truncated
            surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages_flat_truncated
            
            # Compute losses
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Extract the value of the last token in each sequence
            values_last = values[:, -1].view(-1)
            critic_loss = nn.MSELoss()(values_last, returns_flat_truncated)
            
            entropy = dist.entropy().mean()
            
            # Total loss
            loss = actor_loss + self.value_coef * critic_loss - self.entropy_coef * entropy
            
            # Backpropagation
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


### PPO关键函数解读
接下来重点讲解`compute_advantages`和`forward`两个函数，将代码与PPO算法原理和公式对照解读。

#### `compute_advantages` 函数

该函数实现了 **优势函数** $A_t$ 和 **回报函数** $R_t$ 的计算，利用广义优势估计（GAE）来平滑优势函数，减少估计方差。

- **优势函数** $A_t$ 是通过**时间差分误差**（TD-error）来计算的，公式为：
  $$
  \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
  $$
  这部分代码计算了时间差分误差 $\delta_t$，并利用 GAE 进行平滑，即：
  $$
  A_t = \delta_t + (\gamma \lambda) \delta_{t+1} + (\gamma \lambda)^2 \delta_{t+2} + \cdots
  $$
  最终计算出每个时间步的优势 $A_t$ 和回报 $R_t$。


In [ ]:
def compute_advantages(self, rewards, values, gamma=0.99, lambda_=0.95):
    advantages = torch.zeros_like(rewards)
    last_advantage = 0

    for t in reversed(range(len(rewards))):
        if t + 1 < len(values):
            delta = rewards[t] + gamma * values[t + 1] - values[t]
        else:
            delta = rewards[t] - values[t]
        
        advantages[t] = delta + gamma * lambda_ * last_advantage
        last_advantage = advantages[t]
    
    returns = advantages + values[:len(advantages)]
    return advantages, returns

#### `update` 函数
该函数负责根据计算出的 **优势函数** $A_t$ 和 **回报** $R_t$ 更新策略网络的参数。具体地，`update` 函数实现了 PPO 的目标函数，即通过计算 **剪切目标函数**（clipped surrogate objective）来进行优化，并计算出 **策略损失**、**价值损失** 和 **熵损失**。


1. **新旧策略概率比 $r_t(\theta)$**：
   - 在代码中，`new_log_probs` 表示新策略的对数概率，`old_log_probs_flat_truncated` 表示旧策略的对数概率。
   - 通过比值 $r_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$，我们计算了新旧策略的比率。

2. **计算目标损失函数**：
   - PPO 中的目标函数是 $min(r_t(\theta) A_t, clip(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t)$。在代码中，这个部分通过 `torch.min(surr1, surr2)` 来计算。`surr1` 是原始的 $r_t(\theta) A_t$，`surr2` 是经过剪切后的目标 $\text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) A_t$。

3. **价值损失（Critic Loss）**：
   - 价值损失是通过均方误差（MSE）来计算的，公式为：
     $$
     L_{\text{critic}} = (V_{\theta}(s_t) - R_t)^2
     $$
     在代码中是 `critic_loss = nn.MSELoss()(values_last, returns_flat_truncated)`，这里 `values_last` 是模型预测的最后一个时间步的状态值，`returns_flat_truncated` 是对应的回报。

4. **熵损失（Entropy Loss）**：
   - 熵损失鼓励策略保持随机性以促进探索，代码中通过 `dist.entropy().mean()` 计算熵。

5. **总损失**：
   - 最终的总损失是演员损失（actor loss）、价值损失（critic loss）和熵损失（entropy loss）的加权和：
     $$
     L^{PPO}(\theta) = L^{CLIP}(\theta) - c_1 \mathbb{E}_t \left[ (V_{\theta}(s_t) - R_t)^2 \right] + c_2 \mathbb{E}_t \left[ H(\pi_{\theta}(\cdot|s_t)) \right]
     $$
     在代码中是 `loss = actor_loss + self.value_coef * critic_loss - self.entropy_coef * entropy`。

6. **反向传播与更新**：
   - 计算完总损失后，通过 loss.backward() 和 self.optimizer.step() 来进行反向传播和参数更新

In [ ]:
def update(self, states, actions, old_log_probs, rewards, old_values):
    # Reshape states if necessary
    if states.dim() == 2:
        batch_size, seq_len = states.shape
        states = states.unsqueeze(0)  # Add a dimension to make it [1, batch_size, seq_len]
    else:
        num_steps, batch_size, seq_len = states.shape

    # Flatten other tensors
    actions_flat = actions.view(-1)
    old_log_probs_flat = old_log_probs.view(-1)
    advantages, returns = self.compute_advantages(rewards, old_values)
    advantages_flat = advantages.view(-1)
    returns_flat = returns.view(-1)

    for _ in range(5):  # PPO epochs
        logits, _, values = self.model(states.view(-1, seq_len))
        
        # Focus on the logits of the last token in the sequence
        next_token_logits = logits[:, -1, :]
        new_probs = F.softmax(next_token_logits, dim=-1)
        dist = Categorical(new_probs)

        # Ensure actions_flat matches the shape of new_probs
        actions_flat_truncated = actions_flat[:new_probs.size(0)]
        old_log_probs_flat_truncated = old_log_probs_flat[:new_probs.size(0)] # 旧策略的对数概率
        advantages_flat_truncated = advantages_flat[:new_probs.size(0)]
        returns_flat_truncated = returns_flat[:new_probs.size(0)]

        # 计算新策略的对数概率
        new_log_probs = dist.log_prob(actions_flat_truncated)

        # Calculate probability ratio
        ratio = torch.exp(new_log_probs - old_log_probs_flat_truncated)
        surr1 = ratio * advantages_flat_truncated
        surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages_flat_truncated

        # 计算目标损失函数
        actor_loss = -torch.min(surr1, surr2).mean()

        # 提取在每个序列中最后一个token的值
        values_last = values[:, -1].view(-1)
        critic_loss = nn.MSELoss()(values_last, returns_flat_truncated)

        entropy = dist.entropy().mean()

        # 计算总的损失
        loss = actor_loss + self.value_coef * critic_loss - self.entropy_coef * entropy

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

## `compute_reward`奖励计算函数
`compute_reward` 函数的目的是为生成的文本或推理步骤提供反馈奖励，这种反馈可以分为两部分：
- **过程级奖励**：基于生成的文本是否满足一定的质量标准（例如，是否包含关键的推理过程或答案的某些部分）。
- **结果级奖励**：根据最终的推理结果与目标结果之间的差距来给出奖励。

- **输入**：
  - `state`: 一个张量，表示当前模型的状态。假设 `state` 的形状为 `[batch_size, sequence_length]`，其中 `state[:, -1]` 是最新生成的 token（假设模型是基于序列生成的）。
  - `target_result`: 目标结果，通常是一个数字，表示理想答案或目标值。


`compute_reward` 函数的工作流程非常清晰，主要是通过以下几个步骤：
1. **提取生成的文本**：从生成的 token 中获取生成的文本。
2. **检查是否包含结果**：根据文本是否包含 `"result is"` 来判断生成结果是否有效。
3. **计算奖励**：根据生成的结果与目标结果之间的误差计算奖励。如果答案完全正确，给出高奖励；如果接近正确答案，给出中等奖励；如果远离目标，给出低奖励；没有有效结果则给中立奖励。
4. **异常处理**：对于无法处理的输出或错误输出，给予惩罚。

In [19]:
def compute_reward(state, target_result):
    # 这一行代码从 `state` 中提取出每个样本的最后一个生成 token（假设 `state` 是一个包含多个时间步的序列数据）。`state[:, -1]` 取出最后一个时间步的生成 token。
    generated_tokens = state[:, -1].cpu().numpy() 
    rewards = []
    for tokens in generated_tokens:
        try:
            # 对于每个生成的 `tokens`，调用 `detokenize` 函数将 tokens 转换为生成的文本
            generated_text = detokenize(tokens)
            # 检查生成的文本中是否包含 "result is"，这是生成模型用于标识答案或推理结果的标志。如果存在，则截取 "result is" 后面的部分作为结果字符串。
            # 提取 "result is" 后面的字符串并去掉首尾空白字符。
            if "result is" in generated_text:
                result_str = generated_text.split("result is")[-1].strip()
                # 将提取出来的结果字符串转换为数字。如果 `result_str` 是整数，转为 `int` 类型；否则，转为 `float` 类型。
                result = int(result_str) if result_str.isdigit() else float(result_str)
                """
                结果级奖励：根据生成的结果与目标结果之间的差距来分配奖励：
                - 如果生成结果与目标结果的差距小于 1e-6（考虑到浮动误差），则奖励为 1.0，表示完全正确。
                - 如果差距小于 5，则奖励为 0.5，表示较为接近。
                - 如果差距小于 10，奖励为 0.2，表示有一定接近。
                - 如果差距大于 10，则奖励为 -0.2，表示较为远离正确答案。
                """
                if abs(result - target_result) < 1e-6:  # Allow for small floating-point differences
                    rewards.append(1.0)
                elif abs(result - target_result) < 5:  # Close answer
                    rewards.append(0.5)
                elif abs(result - target_result) < 10:  # Somewhat close answer
                    rewards.append(0.2)
                else:
                    rewards.append(-0.2)
            else:
                # 如果生成的文本中没有包含 "result is"，则说明模型没有给出完整的答案或没有给出有效的结果，在这种情况下返回中立奖励 0.0，表示没有好的表现，但也不做惩罚。
                rewards.append(0.0)  # Neutral reward for incomplete answers
        except:
            # 如果在处理过程中发生任何错误（例如，detokenize 过程失败或无法正确转换），则会进入 except 语句块，返回一个负奖励 -0.5，表示惩罚模型输出不正确的结果或无效的输出。
            rewards.append(-0.5)  # Penalize malformed outputs
    return torch.tensor(rewards)

### `generate_arithmetic_problem` 函数解读

`generate_arithmetic_problem` 函数的主要目的是生成一个简单的算术问题，并返回相应的问题描述和答案。它通过随机选择运算符（加法、减法、乘法、除法）以及随机生成的操作数来生成问题。该函数的特点是避免除法操作中的除零错误，确保生成有效的算式。

In [20]:
def generate_arithmetic_problem():
    operations = ['+', '-', '*', '/']
    op = random.choice(operations)
    
    while True:
        if op in ['+', '-']:
            a, b = random.randint(1, 100), random.randint(1, 100)
        else:
            a, b = random.randint(1, 10), random.randint(1, 10)
        
        if op == '+':
            result = a + b
            problem = f"Calculate the sum of {a} and {b}"
        elif op == '-':
            result = a - b
            problem = f"Calculate the difference between {a} and {b}"
        elif op == '*':
            result = a * b
            problem = f"Calculate the product of {a} and {b}"
        else:
            if b != 0:  # Avoid division by zero
                result = a // b
                problem = f"Calculate the quotient of {a} and {b}"
            else:
                continue  # Try again if b is zero
        
        if problem and result:
            return problem, result

## 生成推理链
`generate_reasoning_chain` 函数的主要目的是根据一个给定的算术问题和它的答案生成一个推理链（reasoning chain）。推理链是通过逐步解释如何得出最终答案的过程，它详细描述了解决问题的每一步，目的是帮助理解如何通过各个步骤得出结果。

In [21]:
# Generate reasoning chain
def generate_reasoning_chain(problem, result):
    words = problem.split()
    operation = words[3]  # "sum", "difference", "product", or "quotient"
    
    if operation == "sum":
        a, b = map(int, words[-3::2])
        chain = f"Step: First, we identify the numbers: {a} and {b}. "
        chain += f"Next, we add these numbers: {a} + {b}. "
        chain += f"Finally, we get the result: The sum is {result}."
    elif operation == "difference":
        a, b = map(int, words[-3::2])
        chain = f"Step: First, we identify the numbers: {a} and {b}. "
        chain += f"Next, we subtract the second number from the first: {a} - {b}. "
        chain += f"Finally, we get the result: The difference is {result}."
    elif operation == "product":
        a, b = map(int, words[-3::2])
        chain = f"Step: First, we identify the numbers: {a} and {b}. "
        chain += f"Next, we multiply these numbers: {a} * {b}. "
        chain += f"Finally, we get the result: The product is {result}."
    else:  # quotient
        a, b = map(int, words[-3::2])
        chain = f"Step: First, we identify the numbers: {a} and {b}. "
        chain += f"Next, we divide the first number by the second: {a} / {b}. "
        chain += f"Finally, we get the result: The quotient is {result}."
    
    return chain

## 收集推理轨迹
`collect_trajectories` 函数的目的是生成一批训练样本（轨迹）。轨迹是指从初始状态到最终状态的完整序列，它包括模型所生成的所有状态、动作、奖励、对数概率和价值。在强化学习中，这些轨迹用于训练策略模型，从而优化模型的决策过程。

- **输入**：
  - `model`：强化学习中的策略网络或生成模型。它接受状态输入并输出相应的动作和价值估计。
  - `batch_size`：每次采样的轨迹数量，控制每次生成的训练样本数量。

`collect_trajectories` 函数通过以下步骤生成训练数据：

1. **生成算术问题和推理链**：每次生成一个算术问题及其推理链。
2. **标记化问题和推理链**：将问题和推理链转化为模型输入和目标输出。
3. **生成轨迹**：通过模型采样动作，更新状态，计算奖励，并记录相关信息。
4. **返回批次数据**：生成多个轨迹，并将它们合并为一个批次，供强化学习模型进行训练。

#### 重要变量
- **`states`**：存储每个轨迹中的状态序列。
- **`actions`**：存储每个轨迹中的动作序列。
- **`rewards`**：存储每个轨迹中的奖励序列。
- **`log_probs`**：存储每个轨迹中每个动作的对数概率。
- **`values`**：存储每个轨迹中每个状态的值函数（价值估计）。

- **`max_state_length`**：定义了生成状态的最大长度。这里限制了轨迹的最大长度为40步。


In [ ]:
states = []
actions = []
rewards = []
log_probs = []
values = []

max_state_length = 40

#### 1. 生成算术问题和推理链并Tokenize
在每次迭代中：
  - **`generate_arithmetic_problem()`**：生成一个算术问题（例如加法、减法、乘法或除法问题），并计算出正确答案 `result`。
  - **`generate_reasoning_chain()`**：根据生成的问题和答案，生成一个详细的推理链，解释如何得出这个答案。

In [ ]:
problem, result = generate_arithmetic_problem()
reasoning_chain = generate_reasoning_chain(problem, result)

input_ids = torch.tensor([tokenize(problem)])
target_ids = torch.tensor([tokenize(reasoning_chain)])

#### 2. 初始化状态和动作序列

In [ ]:
state = input_ids
action_sequence = torch.full((1, max_state_length), vocab['<pad>'], dtype=torch.long)

#### 3. 生成动作序列
**`for t in range(max_state_length)`**：遍历最大状态长度 `max_state_length`，生成每个时间步的动作。
**状态裁剪和填充**：
  - 如果当前状态长度大于 `max_state_length`，则截断它。
  - 如果当前状态长度小于 `max_state_length`，则通过填充 `<pad>` 符号扩展它，确保状态的长度不超过最大值。

In [ ]:
# for t in range(max_state_length):
if state.size(1) > max_state_length:
    state = state[:, :max_state_length]
elif state.size(1) < max_state_length:
    padding = torch.full((1, max_state_length - state.size(1)), vocab['<pad>'], dtype=state.dtype)
    state = torch.cat([state, padding], dim=1)

#### 4. 模型推理和采样动作
- **`with torch.no_grad()`**：关闭梯度计算，以减少内存消耗，推理时不需要反向传播。
- **`model(state)`**：通过模型计算当前状态下的 logits（未归一化的预测值），以及状态的价值估计（`value`）。
- **`F.softmax(logits[:, -1, :], dim=-1)`**：对最后一个时间步的 logits 使用 softmax，得到动作的概率分布。
- **`Categorical(probs)`**：根据计算得到的概率分布，创建一个 `Categorical` 分布对象，用于从中采样动作。
- **`dist.sample()`**：从分布中采样一个动作。
- **`dist.log_prob(action)`**：计算所采样动作的对数概率，用于后续的强化学习算法（例如 PPO 中的策略更新）。


In [ ]:
with torch.no_grad():
    logits, _, value = model(state)
    probs = F.softmax(logits[:, -1, :], dim=-1)
    dist = Categorical(probs)
    action = dist.sample()
    log_prob = dist.log_prob(action)

#### 5. 更新状态和存储相关信息
- **更新动作序列**：将当前动作添加到 `action_sequence` 中。
- **存储对数概率和价值**：将当前动作的对数概率和状态的价值（`value`）保存到 `log_probs` 和 `values` 列表中。
- **更新状态**：将当前状态的最后一列（动作）替换为新的动作，形成下一个状态。
- **计算奖励**：调用 `compute_reward(state, result)` 函数，根据当前状态和目标结果计算奖励，并将奖励保存到 `rewards` 列表中。

In [ ]:
action_sequence[0, t] = action.item()
log_probs.append(log_prob)
values.append(value[:, -1])

state = torch.cat([state[:, :-1], action.unsqueeze(1)], dim=1)

reward = compute_reward(state, result)
rewards.append(reward)

In [22]:
# Modify collect_trajectories to use arithmetic problems
def collect_trajectories(model, batch_size):
    states = []
    actions = []
    rewards = []
    log_probs = []
    values = []

    max_state_length = 40

    for _ in range(batch_size):
        problem, result = generate_arithmetic_problem()
        reasoning_chain = generate_reasoning_chain(problem, result)
        
        input_ids = torch.tensor([tokenize(problem)])
        target_ids = torch.tensor([tokenize(reasoning_chain)])
        
        state = input_ids
        action_sequence = torch.full((1, max_state_length), vocab['<pad>'], dtype=torch.long)

        for t in range(max_state_length):
            if state.size(1) > max_state_length:
                state = state[:, :max_state_length]
            elif state.size(1) < max_state_length:
                padding = torch.full((1, max_state_length - state.size(1)), vocab['<pad>'], dtype=state.dtype)
                state = torch.cat([state, padding], dim=1)

            with torch.no_grad():
                logits, _, value = model(state)
                probs = F.softmax(logits[:, -1, :], dim=-1)
                dist = Categorical(probs)
                action = dist.sample()
                log_prob = dist.log_prob(action)

            action_sequence[0, t] = action.item()
            log_probs.append(log_prob)
            values.append(value[:, -1])

            state = torch.cat([state[:, :-1], action.unsqueeze(1)], dim=1)

            reward = compute_reward(state, result)
            rewards.append(reward)

            if action.item() == vocab['<eos>']:
                break

        states.append(state)
        actions.append(action_sequence)

    states = torch.cat(states, dim=0)
    actions = torch.cat(actions, dim=0)
    rewards = torch.cat(rewards, dim=0)
    log_probs = torch.cat(log_probs, dim=0)
    values = torch.cat(values, dim=0)

    return states, actions, rewards, log_probs, values


## 监督微调损失函数
supervised_finetuning_loss 函数的作用是计算 监督微调（Supervised Fine-Tuning） 损失，它用于通过交叉熵损失函数来优化模型在给定目标动作（actions）上的表现。这个损失函数通常用于在预训练模型基础上进行微调时，以训练数据中的真实标签（即目标动作）为参考，来更新模型的权重。

In [23]:

def log_metrics(metrics, epoch):
    print(f"Epoch {epoch} Metrics: {metrics}")

def supervised_finetuning_loss(model, batch):
    states, actions = batch
    logits, _ = model(states, generate_reasoning=False)
    
    # Reshape logits to [batch_size * sequence_length, vocab_size]
    batch_size, seq_length, vocab_size = logits.shape
    logits = logits.view(-1, vocab_size)
    
    # Reshape actions to [batch_size * sequence_length]
    target_ids = actions.view(-1)
    
    # Ensure logits and target_ids have the same length
    min_length = min(logits.size(0), target_ids.size(0))
    logits = logits[:min_length]
    target_ids = target_ids[:min_length]
    
    # Compute loss only on non-padded tokens
    non_pad_mask = target_ids != vocab['<pad>']
    logits = logits[non_pad_mask]
    target_ids = target_ids[non_pad_mask]
    
    loss = F.cross_entropy(logits, target_ids)
    return loss

## `evaluate_model` 函数
`evaluate_model` 函数的主要目的是评估给定模型的性能，尤其是对算术问题的推理能力。它在验证模式下运行模型，并通过计算奖励来衡量模型在一批样本上的表现。奖励的计算基于模型生成的推理链与目标结果的接近程度。


`evaluate_model` 函数的作用是：
- 在不进行梯度计算的情况下评估模型在算术推理任务上的性能。
- 对于每个算术问题，使用模型生成推理链，并计算该推理链的奖励。
- 将所有有效样本的奖励累加，并计算平均奖励。
- 输出评估结果，包括平均奖励和有效样本数量。


In [24]:
def evaluate_model(model, batch_size):
    model.eval() # 将模型切换到评估模式，关闭诸如 dropout 或 batch normalization 等只在训练阶段启用的机制，从而使得评估时模型的行为更加稳定。
    total_reward = 0 # 用于累加所有有效样本的奖励值。
    valid_samples = 0 # 用于计数有效样本的数量，这有助于在最后计算平均奖励时避免除以零的错误。
    # 表示在此代码块内，不会进行梯度计算。这是因为评估过程中不需要进行反向传播，关闭梯度计算可以节省内存和计算资源。
    with torch.no_grad():
        # 遍历给定的批次大小 `batch_size`。对于每一个样本，生成一个算术问题并评估模型的推理能力。
        for _ in range(batch_size):
            try:
                # 会随机生成一个算术问题，并计算出其正确的答案 `result`。
                problem, result = generate_arithmetic_problem()
                input_ids = torch.tensor([tokenize(problem)])
                
                # 这部分代码确保生成的 `input_ids` 不为空。如果为空，说明 `tokenize` 过程中可能出现了问题，因此会跳过该样本，并打印警告信息。
                if input_ids.numel() == 0:
                    print(f"Warning: Empty input tensor for problem: {problem}")
                    continue
                
                # 调用模型生成一个推理链，模型根据给定的算术问题 `input_ids` 推理并生成 `completion_tokens`（即推理链的标记），`reasoning_tokens` 和 `subtasks` 分别可能是推理过程中的中间步骤和子任务（具体依赖于模型的实现）。
                completion_tokens, reasoning_tokens, subtasks = model.generate_completion(input_ids, max_new_tokens=50) # `max_new_tokens=50` 设置了推理链的最大长度。
                
                # 如果模型成功生成了推理链（`completion_tokens` 不为空），则使用 `compute_reward` 函数计算该推理链的奖励。奖励是根据生成的推理链与目标结果 `result` 的匹配程度来计算的。
                if completion_tokens:
                    reward = compute_reward(torch.tensor([completion_tokens]), result)
                    total_reward += reward.item() #  奖励被累加到 `total_reward` 中，并且有效样本计数 `valid_samples` 增加。
                    valid_samples += 1
                else:
                    print(f"Warning: Empty output for problem: {problem}")
            except Exception as e:
                print(f"Error during evaluation: {e}")
    
    model.train()  # 将模型重新设置为训练模式
    avg_reward = total_reward / valid_samples if valid_samples > 0 else 0 # 计算平均奖励
    return {"average_reward": avg_reward, "valid_samples": valid_samples}

In [25]:
def adjust_problem_difficulty(epoch):
    # Implement dynamic difficulty adjustment based on model performance
    global problem_difficulty
    if epoch < 100:
        problem_difficulty = "easy"
    elif epoch < 300:
        problem_difficulty = "medium"
    else:
        problem_difficulty = "hard"

## 开始训练
`train_o1_model` 是一个训练函数，它使用了强化学习和监督学习的混合策略来训练一个模型（通常是一个生成式模型），并通过 PPO（Proximal Policy Optimization）算法优化模型的推理能力。它结合了基于奖励的强化学习训练和基于标签的监督学习训练，提升了模型在算术推理任务中的表现。

In [26]:
def train_o1_model(model, optimizer, num_epochs, batch_size):
    ppo = PPO(model, optimizer)
    
    for epoch in range(num_epochs):
        # Generate a batch of arithmetic problems
        states, actions, rewards, old_log_probs, values = collect_trajectories(model, batch_size)
        
        # Supervised learning step
        sl_loss = supervised_finetuning_loss(model, (states, actions))
        optimizer.zero_grad()
        sl_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    
        # Reinforcement learning step
        ppo.update(states, actions, old_log_probs, rewards, values)
    
        # Evaluation and logging
        if epoch % 10 == 0:
            metrics = evaluate_model(model, batch_size)
            log_metrics(metrics, epoch)

        print(f'Epoch {epoch} completed')
            
        # Dynamic curriculum learning
        if epoch % 50 == 0:
            adjust_problem_difficulty(epoch)

In [27]:
d_model = 128
nhead = 8
num_layers = 4
dropout = 0.1

# Initialize the model
model = O1Model(vocab_size, d_model, nhead, num_layers)
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Training parameters
num_epochs = 500
batch_size = 64

# Train the model
train_o1_model(model, optimizer, num_epochs, batch_size)

# Save the model
torch.save(model.state_dict(), "o1_model.pth")